# Malinois MPRA Activity Scoring

![Malinois MPRA Activity Scoring](https://proto-bio.github.io/proto-assets/images/tool/malinois/hero.png)

Malinois is the regulatory-sequence convolutional neural network at the core of CODA (Computational Optimization of DNA Activity), published by Gosai et al. (2024). It predicts massively parallel reporter assay (MPRA) activity for 200 bp DNA inserts in three cell contexts — K562, HepG2, and SK-N-SH — and was trained on aggregated log2 fold-change measurements to enable machine-guided design of cell-type-targeting cis-regulatory elements.

This notebook demonstrates both Malinois tools: `run_malinois_score`, which returns reverse-complement-averaged raw activity for candidate inserts, and `run_malinois_gradient`, which reports each requested cell type's raw activity and its gradient with respect to relaxed `B x L x 4` DNA logits for Fast SeqProp-style sequence design.

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_overview,
    display_api_reference,
    display_docs_section,
    display_doc_link,
    display_available_tools,
)

display_doc_link("malinois")
display_overview("malinois")
display_docs_section("malinois", "Background")

## Available tools

In [ ]:
display_available_tools("malinois")

### `run_malinois_score`

Given one or more 200 bp DNA inserts, this function adds the MPRA assay flanks expected by the published checkpoint, runs the forward and reverse-complement sequences through Malinois, and returns the averaged raw activity for each requested cell type. It is the tool to use when ranking or screening regulatory DNA candidates by predicted K562, HepG2, or SK-N-SH activity. Scoring is deterministic and batched, so `batch_size` changes throughput without changing results.

In [ ]:
from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisScoreConfig,
    MalinoisScoreInput,
    run_malinois_score,
)

In [ ]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_score")

# Two real 200 bp MPRA inserts from Gosai et al. 2024 (Table S2): a strongly
# active enhancer (high measured activity in all three cell types) and a
# near-silent element (measured activity near zero).
active_enhancer = (
    "CCCACCGGGAGCCTCGGAAGGACCGGCCTCCCCTCACTCTAAGCATGCGCGTTAAAGCGAC"
    "AGCATAGGCTATAGAAAGCTTTATGAGAATAGCTGGGAGCCATACAGCCGCAGGGCCGCGT"
    "GGCCTAATGGATAAGGCGTCTGATTCCGGATCAGAAGATTGAGGGTTCGAGTCCCTTCGTG"
    "GTCGTTGCCATGTTAAC"
)
inactive_element = (
    "CATAAACAATTGCTATTCACAAATTATTCGAATCAAATCAGGTCTCAGCAAATTCCCAACT"
    "CTGCAATGTCATTGCAATGTTATATATAAAAGAGATCCTGAAATAAGTAAGCTATTATCAT"
    "ATATTTGAAAATTCGGCCAGGCATGATAGCATGCGCCTGTTATCCCAGCTACTCAGGCTGG"
    "GAAGGCTGAAGTGAGAG"
)
inputs = MalinoisScoreInput(sequences=[active_enhancer, inactive_element])

In [ ]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_score")

config = MalinoisScoreConfig(
    cell_types=["K562", "HepG2", "SKNSH"],
    batch_size=2,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
result = run_malinois_score(inputs, config)

# Malinois recovers the measured contrast: high predicted activity for the
# active enhancer, near-zero for the inactive element.
labels = ["active_enhancer", "inactive_element"]
[{"insert": labels[idx], **item.scores} for idx, item in enumerate(result.results)]

### `run_malinois_gradient`

For gradient-based DNA design, this function takes batched relaxed logits of shape `(B, L, 4)` in `A,C,G,T` order, relaxes them to nucleotide probabilities at the given softmax temperature, and reports one term per requested cell type, each carrying that cell's raw predicted activity. A single call therefore supplies both sides of an on-target / off-target trade-off, and the caller composes a goal per term. With `compute_gradient=True` each term also carries that activity's gradient, with the same shape as the input logits, ready to drive an optimizer loop.

In [ ]:
import numpy as np

from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisActivityTerm,
    MalinoisGradientConfig,
    MalinoisGradientInput,
    run_malinois_gradient,
)

In [ ]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_gradient")

# A batch of two random relaxed 200 bp candidates (B, L, 4) in A,C,G,T order
rng = np.random.default_rng(0)
logits = rng.normal(size=(2, 200, 4)).tolist()
gradient_inputs = MalinoisGradientInput(inputs=logits, temperature=1.0)

In [ ]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_gradient")

# Differentiate the on-target K562 activity and the off-target HepG2 activity
gradient_config = MalinoisGradientConfig(
    gradient_terms=[
        MalinoisActivityTerm(cell_type="K562"),
        MalinoisActivityTerm(cell_type="HepG2"),
    ],
    compute_gradient=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
gradient_result = run_malinois_gradient(gradient_inputs, gradient_config)
# One result per candidate, each with one raw activity and gradient per requested cell
{
    "raw_scores": [
        {term.term.cell_type: round(term.term_value, 3) for term in r.terms}
        for r in gradient_result.results
    ],
    "gradient_shape": np.array(gradient_result.results[0].terms[0].gradient).shape,
}